In [5]:
source(here::here("settings.R"))
source(here::here("utils.R"))

In [18]:
# I/O
io$sce <- file.path(io$basedir, 'results/rna/pseudobulk/celltype/SingleCellExperiment_pseudobulk.rds')
io$outdir <-  file.path(io$basedir,"results/rna_atac/gene_regulatory_networks/pseudobulk/test"); dir.create(io$outdir, showWarnings = F)
# io$chip_GRN.df <- file.path(io$outdir,'chip_GRN_df.csv')
io$tf2gene_virtual_chip <- file.path(io$basedir,"results/rna_atac/virtual_chipseq/pseudobulk/CISBP/TF2gene_after_virtual_chip.txt.gz")

In [7]:
# Minimum in silico chip score
opts$min_chip_score <- 0.25

# Maximum genomic distance
opts$max_distance <- 5e4

# Number of cores for parallel processing
opts$ncores <- 4

In [13]:
####################################################
## Load TF2gene links based on in silico ChIP-seq ##
####################################################

tf2gene_chip.dt <- fread(io$tf2gene_virtual_chip) %>%
  .[chip_score>=opts$min_chip_score & dist<=opts$max_distance] %>% 
  .[,c("tf","gene")] %>% unique # Only keep TF-gene links

In [14]:
head(tf2gene_chip.dt)

tf,gene
<chr>,<chr>
AHR,Dse
AHR,
AHR,Tspyl1
AHR,Nt5dc1
AHR,Gm26564
AHR,Tspyl4


In [15]:
# Options
opts$celltypes <- setdiff(opts$celltypes, c("Visceral_endoderm","ExE_endoderm","ExE_ectoderm","Parietal_endoderm"))

In [19]:
sce <- readRDS(io$sce)[,opts$celltypes]

In [20]:
sce

class: SingleCellExperiment 
dim: 32285 33 
metadata(2): agg_pars n_cells
assays(2): counts logcounts
rownames(32285): Xkr4 Gm1992 ... AC234645.1 AC149090.1
rowData names(0):
colnames(33): Epiblast Primitive_Streak ... Spinal_cord
  Surface_ectoderm
colData names(0):
reducedDimNames(0):
mainExpName: NULL
altExpNames(0):

In [21]:
##########################
## Filter TFs and genes ##
##########################

TFs <- intersect(unique(tf2gene_chip.dt$tf),toupper(rownames(sce)))
genes <- intersect(unique(tf2gene_chip.dt$gene),rownames(sce))

tf2gene_chip.dt <- tf2gene_chip.dt[tf%in%TFs & gene%in%genes,]

# Fetch RNA expression matrices
rna_tf.mtx <- logcounts(sce)[str_to_title(unique(tf2gene_chip.dt$tf)),]; rownames(rna_tf.mtx) <- toupper(rownames(rna_tf.mtx))
rna_targets.mtx <- logcounts(sce)[unique(tf2gene_chip.dt$gene),]

# TO-DO: FILTER OUT LOWLY VARIABLE GENES
rna_tf.mtx <- rna_tf.mtx[apply(rna_tf.mtx,1,var)>0.1,]
rna_targets.mtx <- rna_targets.mtx[apply(rna_targets.mtx,1,var)>0.1,]

TFs <- intersect(unique(tf2gene_chip.dt$tf),rownames(rna_tf.mtx))
genes <- intersect(unique(tf2gene_chip.dt$gene),rownames(rna_targets.mtx))
tf2gene_chip.dt <- tf2gene_chip.dt[tf%in%TFs & gene%in%genes,]

In [22]:
head(tf2gene_chip.dt)

tf,gene
<chr>,<chr>
AHR,Dse
AHR,Nt5dc1
AHR,Tspyl4
AHR,Ccdc88c
AHR,A630072L19Rik
AHR,Abcg1


In [23]:
nrow(tf2gene_chip.dt)

[1] 576805